# Evaluation Pipeline

Runs **4 experiments** with resume support, then produces comparative analysis:

| # | Dataset  | Poisoning |
|---|----------|-----------|
| 1 | HotpotQA | Clean     |
| 2 | HotpotQA | Poisoned  |
| 3 | FEVER    | Clean     |
| 4 | FEVER    | Poisoned  |

## 1 · Setup & Configuration

In [ ]:
import sys, json, random, types
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
from tqdm.notebook import tqdm

ROOT = Path('..').resolve()   # notebooks/ -> project root
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

In [ ]:
from src.eval.evaluate import (
    normalize_answer, exact_match, token_f1, sp_metrics,
    extract_final_answer,
    build_pipeline as build_hotpotqa_pipeline,
)
from src.eval.evaluate_fever import (
    normalize_label, label_accuracy, evidence_metrics, fever_score,
    extract_final_verdict,
    build_pipeline as build_fever_pipeline,
)

# ── Config ───────────────────────────────────────────────────────────────
CFG = types.SimpleNamespace(
    limit       = 100,
    seed        = 42,
    poison_seed = 42,
    top_k       = 10,
    llm         = 'llama3.2:3b',
    embed_model = 'sentence-transformers/all-MiniLM-L12-v2',
)

RESULTS_DIR  = ROOT / 'results'
HOTPOTQA_VAL = ROOT / 'data' / 'hotpotqa' / 'validation.jsonl'
FEVER_VAL    = ROOT / 'data' / 'fever'    / 'paper_dev.jsonl'
RESULTS_DIR.mkdir(exist_ok=True)

PATHS = {
    'hotpotqa_clean':    RESULTS_DIR / 'hotpotqa_clean.jsonl',
    'hotpotqa_poisoned': RESULTS_DIR / 'hotpotqa_poisoned.jsonl',
    'fever_clean':       RESULTS_DIR / 'fever_clean.jsonl',
    'fever_poisoned':    RESULTS_DIR / 'fever_poisoned.jsonl',
}
print('Config ready. Paths:', {k: str(v) for k, v in PATHS.items()})

## 2 · Run Evaluations

Each runner is resume-safe: already-evaluated records are skipped.

In [ ]:
def run_hotpotqa(
    output_path: Path,
    poison: bool = False,
    poison_seed: int | None = None,
) -> list[dict]:
    """Run HotpotQA evaluation with resume support."""
    output_path = Path(output_path)

    samples = [json.loads(l) for l in HOTPOTQA_VAL.read_text().splitlines() if l.strip()]
    samples = random.Random(CFG.seed).sample(samples, CFG.limit)

    already_done: set[str] = set()
    all_records: list[dict] = []
    if output_path.exists():
        for line in output_path.read_text().splitlines():
            if line.strip():
                rec = json.loads(line)
                already_done.add(rec['id'])
                all_records.append(rec)

    pending = [s for s in samples if s['id'] not in already_done]
    if not pending:
        print(f'  [skip] {output_path.name} already complete ({len(all_records)} records)')
        return all_records

    print(f'  {output_path.name}: {len(already_done)} done, {len(pending)} remaining')

    args = types.SimpleNamespace(
        collection  = 'hotpotqa_passages',
        embed_model = CFG.embed_model,
        top_k       = CFG.top_k,
        llm         = CFG.llm,
        poison      = poison,
        poison_seed = poison_seed,
    )
    pipeline = build_hotpotqa_pipeline(args)

    with open(output_path, 'a', encoding='utf-8') as out_f:
        for sample in tqdm(pending, desc=output_path.stem):
            record: dict = {
                'id':       sample['id'],
                'type':     sample['type'],
                'question': sample['question'],
            }
            try:
                result      = pipeline.answer(sample['question'])
                pred        = extract_final_answer(result['answer'])
                gold        = sample['answer']
                ret_titles  = [s['title'] for s in result['sources']]
                gold_titles = list(set(sample['supporting_facts']['title']))
                sp_p, sp_r, sp_f = sp_metrics(ret_titles, gold_titles)
                record.update({
                    'gold_answer':       gold,
                    'pred_answer':       pred,
                    'em':                exact_match(pred, gold),
                    'f1':                token_f1(pred, gold),
                    'sp_precision':      sp_p,
                    'sp_recall':         sp_r,
                    'sp_f1':             sp_f,
                    'retrieved_titles':  ret_titles,
                    'gold_titles':       gold_titles,
                    'poisoning_enabled': poison,
                    'poisoned_count':    sum(1 for s in result['sources'] if s.get('poisoned')),
                    'poisoned_titles':   [s['title'] for s in result['sources'] if s.get('poisoned')],
                    'error':             None,
                })
            except Exception as e:
                record.update({'error': str(e), 'gold_answer': sample['answer']})

            all_records.append(record)
            out_f.write(json.dumps(record) + '\n')
            out_f.flush()

    print(f'  Saved -> {output_path}')
    return all_records

In [ ]:
def run_fever(
    output_path: Path,
    poison: bool = False,
    poison_seed: int | None = None,
) -> list[dict]:
    """Run FEVER evaluation with resume support."""
    output_path = Path(output_path)

    samples = [json.loads(l) for l in FEVER_VAL.read_text().splitlines() if l.strip()]
    samples = random.Random(CFG.seed).sample(samples, CFG.limit)

    already_done: set[int] = set()
    all_records: list[dict] = []
    if output_path.exists():
        for line in output_path.read_text().splitlines():
            if line.strip():
                rec = json.loads(line)
                already_done.add(rec['id'])
                all_records.append(rec)

    pending = [s for s in samples if s['id'] not in already_done]
    if not pending:
        print(f'  [skip] {output_path.name} already complete ({len(all_records)} records)')
        return all_records

    print(f'  {output_path.name}: {len(already_done)} done, {len(pending)} remaining')

    args = types.SimpleNamespace(
        collection  = 'fever_passages',
        embed_model = CFG.embed_model,
        top_k       = CFG.top_k,
        llm         = CFG.llm,
        poison      = poison,
        poison_seed = poison_seed,
    )
    pipeline = build_fever_pipeline(args)

    with open(output_path, 'a', encoding='utf-8') as out_f:
        for sample in tqdm(pending, desc=output_path.stem):
            record: dict = {
                'id':    sample['id'],
                'claim': sample['claim'],
            }
            try:
                result     = pipeline.answer(sample['claim'])
                pred       = extract_final_verdict(result['answer'])
                gold       = sample['label']
                ret_titles = [s['title'] for s in result['sources']]
                gold_titles = list({
                    piece[2].replace('_', ' ')
                    for ann_set in sample.get('evidence', [])
                    for piece in ann_set
                    if len(piece) >= 3 and piece[2]
                })
                ev_p, ev_r, ev_f = evidence_metrics(ret_titles, gold_titles)
                record.update({
                    'gold_label':        gold,
                    'pred_label':        pred,
                    'pred_normalized':   normalize_label(pred),
                    'label_acc':         label_accuracy(pred, gold),
                    'fever_sc':          fever_score(pred, gold, ret_titles, gold_titles),
                    'ev_precision':      ev_p,
                    'ev_recall':         ev_r,
                    'ev_f1':             ev_f,
                    'retrieved_titles':  ret_titles,
                    'gold_titles':       gold_titles,
                    'poisoning_enabled': poison,
                    'poisoned_count':    sum(1 for s in result['sources'] if s.get('poisoned')),
                    'poisoned_titles':   [s['title'] for s in result['sources'] if s.get('poisoned')],
                    'error':             None,
                })
            except Exception as e:
                record.update({'error': str(e), 'gold_label': sample['label']})

            all_records.append(record)
            out_f.write(json.dumps(record) + '\n')
            out_f.flush()

    print(f'  Saved -> {output_path}')
    return all_records

In [ ]:
print('=' * 60)
print('Running all 4 experiments  (resume-safe)')
print('=' * 60)

print('\n[1/4] HotpotQA — Clean')
run_hotpotqa(PATHS['hotpotqa_clean'], poison=False)

print('\n[2/4] HotpotQA — Poisoned')
run_hotpotqa(PATHS['hotpotqa_poisoned'], poison=True, poison_seed=CFG.poison_seed)

print('\n[3/4] FEVER — Clean')
run_fever(PATHS['fever_clean'], poison=False)

print('\n[4/4] FEVER — Poisoned')
run_fever(PATHS['fever_poisoned'], poison=True, poison_seed=CFG.poison_seed)

print('\nAll experiments complete!')

## 3 · Load Results

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    records = [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]
    return pd.DataFrame([r for r in records if r.get('error') is None])

hq_clean    = load_jsonl(PATHS['hotpotqa_clean']).assign(condition='clean',    dataset='HotpotQA')
hq_poisoned = load_jsonl(PATHS['hotpotqa_poisoned']).assign(condition='poisoned', dataset='HotpotQA')
fv_clean    = load_jsonl(PATHS['fever_clean']).assign(condition='clean',    dataset='FEVER')
fv_poisoned = load_jsonl(PATHS['fever_poisoned']).assign(condition='poisoned', dataset='FEVER')

for name, df in [
    ('HotpotQA Clean',    hq_clean),
    ('HotpotQA Poisoned', hq_poisoned),
    ('FEVER Clean',       fv_clean),
    ('FEVER Poisoned',    fv_poisoned),
]:
    print(f'  {name}: {len(df)} records')

## 4 · HotpotQA Analysis

**Metrics:** Exact Match (EM), Token F1, Supporting-Facts Precision/Recall/F1

In [ ]:
HQ_METRICS = {
    'em':           'EM',
    'f1':           'Token F1',
    'sp_precision': 'SP Precision',
    'sp_recall':    'SP Recall',
    'sp_f1':        'SP F1',
}

hq = pd.concat([hq_clean, hq_poisoned])

hq_summary = (
    hq.groupby('condition')[list(HQ_METRICS)].mean()
      .mul(100).round(1)
      .rename(columns=HQ_METRICS)
      .T
)
hq_summary.columns.name = None
hq_summary.index.name = 'Metric'
hq_summary['Delta (poisoned-clean)'] = (hq_summary['poisoned'] - hq_summary['clean']).round(1)

print('HotpotQA — Clean vs Poisoned (all question types)\n')
display(hq_summary.style
        .format('{:.1f}')
        .background_gradient(cmap='RdYlGn', subset=['clean', 'poisoned'], axis=None))

In [ ]:
hq_type = (
    hq.groupby(['type', 'condition'])[['em', 'f1', 'sp_f1']].mean()
       .mul(100).round(1)
       .rename(columns={'em': 'EM', 'f1': 'Token F1', 'sp_f1': 'SP F1'})
       .unstack('condition')
)
print('HotpotQA — Breakdown by question type\n')
display(hq_type.style.format('{:.1f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (metric, label) in zip(axes, [('em', 'Exact Match (%)'), ('f1', 'Token F1 (%)')]):
    for i, (qtype, color) in enumerate([('bridge', '#4C72B0'), ('comparison', '#DD8452')]):
        vals = hq[hq['type'] == qtype].groupby('condition')[metric].mean().mul(100)
        x_pos = [i * 0.3, i * 0.3 + 1.1]
        bars = ax.bar(x_pos,
                      [vals.get('clean', 0), vals.get('poisoned', 0)],
                      width=0.25, label=qtype.capitalize(), color=color, alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks([0.15, 1.25])
    ax.set_xticklabels(['Clean', 'Poisoned'])
    ax.set_ylabel(label)
    ax.set_title(f'HotpotQA — {label}')
    ax.legend()
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'hotpotqa_clean_vs_poisoned.png', dpi=150)
plt.show()

## 5 · FEVER Analysis

**Metrics:** Label Accuracy, FEVER Score, Evidence Precision/Recall/F1

In [ ]:
FV_METRICS = {
    'label_acc':    'Label Accuracy',
    'fever_sc':     'FEVER Score',
    'ev_precision': 'Ev. Precision',
    'ev_recall':    'Ev. Recall',
    'ev_f1':        'Ev. F1',
}

fv = pd.concat([fv_clean, fv_poisoned])

fv_summary = (
    fv.groupby('condition')[list(FV_METRICS)].mean()
      .mul(100).round(1)
      .rename(columns=FV_METRICS)
      .T
)
fv_summary.columns.name = None
fv_summary.index.name = 'Metric'
fv_summary['Delta (poisoned-clean)'] = (fv_summary['poisoned'] - fv_summary['clean']).round(1)

print('FEVER — Clean vs Poisoned\n')
display(fv_summary.style
        .format('{:.1f}')
        .background_gradient(cmap='RdYlGn', subset=['clean', 'poisoned'], axis=None))

In [ ]:
fv_label = (
    fv.groupby(['gold_label', 'condition'])[['label_acc', 'fever_sc', 'ev_f1']].mean()
       .mul(100).round(1)
       .rename(columns={'label_acc': 'Label Acc', 'fever_sc': 'FEVER Score', 'ev_f1': 'Ev. F1'})
       .unstack('condition')
)
print('FEVER — Breakdown by gold label\n')
display(fv_label.style.format('{:.1f}'))

In [ ]:
label_order = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']
colors = {'SUPPORTS': '#4C72B0', 'REFUTES': '#DD8452', 'NOT ENOUGH INFO': '#55A868'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (metric, label) in zip(axes, [
    ('label_acc', 'Label Accuracy (%)'),
    ('fever_sc',  'FEVER Score (%)'),
]):
    for i, gold_label in enumerate(label_order):
        sub = fv[fv['gold_label'] == gold_label]
        if sub.empty:
            continue
        vals = sub.groupby('condition')[metric].mean().mul(100)
        x_pos = [i * 0.3, i * 0.3 + 1.1]
        bars = ax.bar(x_pos,
                      [vals.get('clean', 0), vals.get('poisoned', 0)],
                      width=0.25, label=gold_label, color=colors[gold_label], alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks([0.3, 1.4])
    ax.set_xticklabels(['Clean', 'Poisoned'])
    ax.set_ylabel(label)
    ax.set_title(f'FEVER — {label}')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fever_clean_vs_poisoned.png', dpi=150)
plt.show()

## 6 · Poisoning Impact — Cross-Dataset Comparison

Δ = poisoned − clean (negative = degradation from poisoning)

In [ ]:
hq_delta = (
    hq.groupby('condition')[['em', 'f1', 'sp_precision', 'sp_recall', 'sp_f1']]
      .mean().mul(100)
      .diff().iloc[1]
      .rename({'em': 'EM', 'f1': 'Token F1',
               'sp_precision': 'SP Precision', 'sp_recall': 'SP Recall', 'sp_f1': 'SP F1'})
)

fv_delta = (
    fv.groupby('condition')[['label_acc', 'fever_sc', 'ev_precision', 'ev_recall', 'ev_f1']]
      .mean().mul(100)
      .diff().iloc[1]
      .rename({'label_acc': 'Label Acc', 'fever_sc': 'FEVER Score',
               'ev_precision': 'Ev. Precision', 'ev_recall': 'Ev. Recall', 'ev_f1': 'Ev. F1'})
)

delta_df = pd.DataFrame({'HotpotQA Delta': hq_delta, 'FEVER Delta': fv_delta})
print('Poisoning impact: Delta = Poisoned - Clean (percentage points)\n')
display(delta_df.style
        .format('{:+.1f}')
        .background_gradient(cmap='RdYlGn_r', axis=None))

In [ ]:
hq_items = [('HQ: EM',       hq_delta.get('EM', 0)),
            ('HQ: Token F1', hq_delta.get('Token F1', 0)),
            ('HQ: SP F1',    hq_delta.get('SP F1', 0))]
fv_items = [('FV: Label Acc',   fv_delta.get('Label Acc', 0)),
            ('FV: FEVER Score', fv_delta.get('FEVER Score', 0)),
            ('FV: Ev. F1',      fv_delta.get('Ev. F1', 0))]

all_items = [(k, v, '#4C72B0') for k, v in hq_items] + \
            [(k, v, '#DD8452') for k, v in fv_items]
labels, values, colors_list = zip(*all_items)

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(range(len(labels)), values, color=colors_list, alpha=0.85, width=0.6)

for bar, val in zip(bars, values):
    offset = 0.3 if val >= 0 else -1.5
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
            f'{val:+.1f}', ha='center', va='bottom', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Delta percentage points (poisoned - clean)')
ax.set_title('Poisoning Impact: HotpotQA vs FEVER')
ax.legend(handles=[Patch(color='#4C72B0', label='HotpotQA'),
                   Patch(color='#DD8452', label='FEVER')])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'poisoning_impact.png', dpi=150)
plt.show()

## 7 · Error Analysis

Identify questions/claims that were answered **correctly in the clean run** but **incorrectly in the poisoned run** — i.e., direct regressions caused by poisoning.

In [ ]:
merge_hq = hq_clean[['id', 'question', 'gold_answer', 'pred_answer', 'em', 'type']].merge(
    hq_poisoned[['id', 'pred_answer', 'em']].rename(
        columns={'pred_answer': 'pred_poisoned', 'em': 'em_poisoned'}),
    on='id', how='inner',
)
flipped_hq = merge_hq[(merge_hq['em'] == 1) & (merge_hq['em_poisoned'] == 0)]

print(f'HotpotQA — correct when clean, wrong when poisoned: {len(flipped_hq)} / {len(merge_hq)}\n')
for _, row in flipped_hq.head(5).iterrows():
    print(f'  [{row["type"]}] {row["question"]}')
    print(f'    Gold:            {row["gold_answer"]}')
    print(f'    Clean answer:    {row["pred_answer"]}')
    print(f'    Poisoned answer: {row["pred_poisoned"]}')
    print()

In [ ]:
merge_fv = fv_clean[['id', 'claim', 'gold_label', 'pred_label', 'label_acc']].merge(
    fv_poisoned[['id', 'pred_label', 'label_acc']].rename(
        columns={'pred_label': 'pred_poisoned', 'label_acc': 'acc_poisoned'}),
    on='id', how='inner',
)
flipped_fv = merge_fv[(merge_fv['label_acc'] == 1) & (merge_fv['acc_poisoned'] == 0)]

print(f'FEVER — correct when clean, wrong when poisoned: {len(flipped_fv)} / {len(merge_fv)}\n')
for _, row in flipped_fv.head(5).iterrows():
    print(f'  [{row["gold_label"]}] {row["claim"]}')
    print(f'    Clean verdict:   {row["pred_label"]}')
    print(f'    Poisoned verdict:{row["pred_poisoned"]}')
    print()